In [ ]:
"""Notebook to analyze ChromScore values for certain regions."""

# pylint: disable=redefined-outer-name, expression-not-assigned, import-error, not-callable, pointless-statement, no-value-for-parameter, unused-argument, use-dict-literal, too-many-lines, too-many-branches, duplicate-code

In [ ]:
%load_ext autoreload
%autoreload 2

## SETUP

In [ ]:
from __future__ import annotations

import copy
import tarfile
from pathlib import Path
from typing import IO, Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display  # pylint: disable=unused-import
from plotly.subplots import make_subplots
from scipy import stats

from epiclass.core.data_source import EpiDataSource  # pylint: disable=unused-import
from epiclass.core.metadata import Metadata
from epiclass.utils.bed_utils import bed_to_bins

ASSAY = "assay_epiclass"
TRACK_TYPE = "track_type"
CELL_TYPE = "harmonized_sample_ontology_intermediate"

In [ ]:
%matplotlib inline

In [ ]:
base = Path.home() / "Projects/epiclass"
input_base = base / "input"
output_base = base / "output"

chromsize_path = input_base / "chromsizes" / "hg38.noy.chrom.sizes"
metadata_path = (
    input_base
    / "metadata/dfreeze-v2/hg38_2023-epiatlas-dfreeze_v2.1_w_encode_noncore_2.json"
)

base_logdir = output_base / "logs"
logdir = base_logdir / "epiatlas-dfreeze-v2.1/hdf5_stats"

if not logdir.exists():
    logdir.mkdir(parents=True)

In [ ]:
paper_dir = output_base / "paper"
table_dir = paper_dir / "tables"

base_data_dir = paper_dir / "data"

In [ ]:
chromsizes: List[Tuple[str, int]] = EpiDataSource.load_external_chrom_file(chromsize_path)

chroms: List[str] = sorted([chrom for chrom, _ in chromsizes])

In [ ]:
metadata = Metadata(metadata_path)
metadata_df = metadata.to_df()

In [ ]:
SHAP_dir = base_data_dir / "SHAP"
if not SHAP_dir.exists():
    raise FileNotFoundError(f"Directory {SHAP_dir} does not exist.")

cell_type_shap_dir = (
    SHAP_dir / "hg38_100kb_all_none" / f"{CELL_TYPE}_1l_3000n" / "10fold-oversampling"
)
beds_file = cell_type_shap_dir / "select_beds_top303.tar.gz"
if not beds_file.exists():
    raise FileNotFoundError(f"File {beds_file} does not exist.")

## Biospecimens important SHAP regions

Read important bins values, and find possible classes for each unique bin.

In [ ]:
ct_important_bins: Dict[str, List[int]] = {}
with tarfile.open(beds_file, "r:gz") as tar:
    for member in tar.getmembers():
        filename = member.name

        if "merge_samplings" in filename and filename.endswith("bed"):
            file_obj: IO[bytes] = tar.extractfile(member)  # type: ignore

            cell_type = (
                filename.split("/")[1]
                .replace("merge_samplings_", "")
                .replace("_features.bed", "")
                .lower()
            )

            ct_important_bins[cell_type] = bed_to_bins(
                file_obj, chroms=chromsizes, resolution=100 * 1000
            )

In [ ]:
all_bins = set()
all_bins_list = []
for bins in ct_important_bins.values():
    all_bins.update(bins)
    all_bins_list.extend(bins)

all_bins = sorted(all_bins)
print(len(all_bins), "unique important bins across cell types.")
print(len(all_bins_list), "total important bins across cell types.")

In [ ]:
# Find relevant cell types for each bin, optimized for pandas future vectorization
relevant_pairs_list = []
for cell_type, bins_list in ct_important_bins.items():
    for bin_idx in bins_list:
        relevant_pairs_list.append({"bin_index": bin_idx, CELL_TYPE: cell_type})

bin_to_relevant_ct_df = pd.DataFrame(relevant_pairs_list)
bin_to_relevant_ct_df["bin_index"] = bin_to_relevant_ct_df["bin_index"].astype(int)

assert bin_to_relevant_ct_df.shape[0] > len(ct_important_bins)

In [ ]:
print(bin_to_relevant_ct_df.shape)
print(bin_to_relevant_ct_df["bin_index"].nunique())
print("\nImportant regions for each cell type:")
print(bin_to_relevant_ct_df[CELL_TYPE].value_counts(dropna=False))

In [ ]:
classifier_cell_types = set(bin_to_relevant_ct_df[CELL_TYPE].unique())
assert len(classifier_cell_types) == 16

## ChromScore hdf5 values

For each cell type important feature, find the average ChromScore value throughout associated cell types.  
If bin is present in multiple classes, just use files for all those classes.

### Read ChromScore values, and map files to their cell type.

Using hdf5 output from `bigwig_metrics.py`.

Maximum ChromScore values computed directly from bigwigs, for each 100kb regions.

In [ ]:
chromscore_dir = paper_dir / "data" / "ChromScore"
chromscore_file = chromscore_dir / "max_metrics_clean.h5"

to_clean = False
if not chromscore_file.exists():
    print(f"{chromscore_file} does not exist")
    chromscore_file = chromscore_dir / "max_metrics.h5"
    to_clean = True

if not chromscore_file.exists():
    raise FileNotFoundError(f"{chromscore_file} does not exist")

print(f"Loading {chromscore_file}")
chromscores_df: pd.DataFrame = pd.read_hdf(chromscore_file)  # type: ignore
print("Chromscores shape", chromscores_df.shape)
display(chromscores_df.head(n=2))

In [ ]:
def transform_chromscore_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean up values in .h5 inplace.

    - If values in .h5 were stored a lists (with one value), get val
    - fillna as 0
    """
    # check for nans
    for col in df.columns:
        if df[col].isna().sum():
            print(col, df[col].isna().sum())

    # check if values are lists
    if isinstance(df.iloc[0, 0], list):
        df = df.map(lambda x: x[0])

    df.fillna(0, inplace=True)

    return df

In [ ]:
if to_clean:
    print("Chromscores shape", chromscores_df.shape)
    chromscores_df = transform_chromscore_df(chromscores_df)
    print("Chromscores shape", chromscores_df.shape)

    chromscore_file = chromscore_dir / "max_metrics_clean.h5"
    chromscores_df.to_hdf(
        chromscore_file,
        key="df",
        mode="w",
        format="fixed",
        complevel=9,
    )

In [ ]:
chromscores_df["epirr"] = chromscores_df.index.str.split(".").str[0]

# Create a mapping from epirr_id_without_version to cell_type
epirr_to_cell_type = dict(
    metadata_df.loc[:, ["epirr_id_without_version", CELL_TYPE]].values
)

chromscores_df[CELL_TYPE] = (
    chromscores_df["epirr"].map(epirr_to_cell_type).str.replace(" ", "_").str.lower()
)
display(chromscores_df[CELL_TYPE].value_counts(dropna=False))

In [ ]:
# Only keep files from classifier 16ct
condition = chromscores_df[CELL_TYPE].isin(classifier_cell_types)
print(
    f"Keeping {condition.sum()} files out of {len(chromscores_df)} from classifier cell types."
)
chromscores_df = chromscores_df[condition]
assert chromscores_df[CELL_TYPE].nunique() == 16

### Find mean metric for each bin (for their relevant files)

In [ ]:
# Melting global chromscores for future operation, now all values are in one column
# Assuming all columns starting with chr are bins, and all 100kb bins are present
region_cols_mapper = {
    col: idx
    for idx, col in enumerate(chromscores_df.columns)
    if isinstance(col, str) and col.startswith("chr")
}
assert len(region_cols_mapper) == 30321

In [ ]:
chromscores_df.rename(columns=region_cols_mapper, inplace=True)  # type: ignore

In [ ]:
melted_chromscores = chromscores_df.reset_index().rename(columns={"index": "filename"})
melted_chromscores = melted_chromscores.melt(
    id_vars=["epirr", CELL_TYPE],
    value_vars=list(region_cols_mapper.values()),
    var_name="bin_index",
    value_name="chromscore_value",
)
melted_chromscores["bin_index"] = melted_chromscores["bin_index"].astype(int)
print(melted_chromscores.shape)
print(melted_chromscores["bin_index"].nunique())
print(melted_chromscores["epirr"].nunique())
display(melted_chromscores.head(n=2))

In [ ]:
# Merge melted chromscores with bin_to_relevant_ct_df, efficient for pandas
# Filters out all irrelevant bins
merged_df = pd.merge(
    melted_chromscores, bin_to_relevant_ct_df, on=["bin_index", CELL_TYPE], how="inner"
)
print(merged_df.shape)
print(merged_df["bin_index"].nunique())
print(merged_df["epirr"].nunique())
display(merged_df.head(n=2))

In [ ]:
# Keep only columns of interest for plotting
merged_df = merged_df[["epirr", CELL_TYPE, "bin_index", "chromscore_value"]]
total_files = merged_df["epirr"].nunique()
print(f"Total files in merged df: {total_files}")

## Plot

Global chromscore metric (max or mean) vs per cell type, for important SHAP regions only

In [ ]:
def prepare_chromscore_per_biospecimen_data(
    selected_bins_df: pd.DataFrame,
    all_chromscores_df: pd.DataFrame,
    cell_types_col: str = CELL_TYPE,
    allowed_cell_types: List[str] | None = None,
) -> Dict[str, Dict[str, List[float] | int]]:
    """
    Prepare plot data for chromscore per biospecimen plot.

    Each biospecimen need values for:
    - important features
    - global distribution

    This is done with independent file subsets.
    """
    grouped_means = {}

    for biospecimen, df in selected_bins_df.groupby(by=cell_types_col):
        if (allowed_cell_types is not None) and (biospecimen not in allowed_cell_types):
            print(f"Skipping {biospecimen}.")
            continue
        print(f"Processing {biospecimen}")

        all_chromscores_group = all_chromscores_df.loc[
            all_chromscores_df[cell_types_col] == biospecimen, :
        ]
        if not all_chromscores_group["chromscore_value"].isna().sum() == 0:
            raise ValueError(f"Missing values in chromscore_value for {biospecimen}")
        nb_files = df["epirr"].nunique()

        # Important features
        avg_per_bin = df.groupby("bin_index")["chromscore_value"].mean().to_list()

        # Global distribution
        all_means_file_subset = (
            all_chromscores_group.groupby("bin_index")["chromscore_value"]
            .mean()
            .to_list()
        )

        grouped_means[biospecimen] = {
            "avg_per_bin": avg_per_bin,
            "all_means_file_subset": all_means_file_subset,
            "nb_files": nb_files,
        }

    grouped_means["all_files"] = (
        all_chromscores_df.groupby("bin_index")["chromscore_value"].mean().to_list()
    )

    return grouped_means

In [ ]:
def cohens_d(x, y):
    """
    Calculate Cohen's d for independent samples.
    Uses pooled standard deviation accounting for unequal variances.
    """
    n1, n2 = len(x), len(y)
    var1, var2 = np.var(x, ddof=1), np.var(y, ddof=1)

    # Pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))

    # Cohen's d
    d = (np.mean(x) - np.mean(y)) / pooled_std

    return d


def test_distribution(
    x: List[float], y: List[float], verbose: bool = True
) -> Tuple[Any, Any, float]:
    """Test for distribution difference. x is used as reference for the number of samples.

    Welch's t-test and Brunner-Munzel test are computed.

    Returns:
        Tuple[TtestResult, BrunnerMunzelResult, float]: test results and cohen's d effect size.
    """
    if verbose:
        print(f"Number of samples in x: {len(x)}")
        print(f"Number of samples in y: {len(y)}")

    Welch_test = stats.ttest_ind(
        a=x,
        b=y,
        equal_var=False,
        alternative="two-sided",
        nan_policy="raise",
    )

    BM_test = stats.brunnermunzel(
        x,
        y,
        alternative="two-sided",
        nan_policy="raise",
        distribution="t",
    )

    effect_size = cohens_d(x, y)
    if verbose:
        print(f"Cohen's d effect size: {effect_size:.4f}")
        print(
            f"Welch's t-test: statistic={Welch_test.statistic:.4f}, pvalue={Welch_test.pvalue:.4e}"
        )
        print(
            f"Brunner-Munzel test: statistic={BM_test.statistic:.4f}, pvalue={BM_test.pvalue:.4e}"
        )

    return Welch_test, BM_test, effect_size


def define_pval_label(pval: float) -> str:
    """Define p-value label."""
    pval_symbol = ""
    if pval < 0.001:
        pval_symbol = "<0.001***"
    elif pval < 0.01:
        pval_symbol = "<0.01**"
    elif pval < 0.05:
        pval_symbol = "<0.05*"
    elif pval >= 0.05:
        pval_symbol = ">0.05 NS"

    return pval_symbol

In [ ]:
def plot_chromscore_per_biospecimen_violin(
    graph_data: Dict[str, Dict[str, List[float] | int]],
    cell_types: List[str] | None = None,
    logdir: Path | None = None,
    do_subplots: bool = True,
    filename: str = "important_features_16ct_max_chromscore_100kb_per_biospecimen_2violin",
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Plot average of 'max chromscore' per biospecimen as violin plots,
    using regions and files per biospecimen independently.

    The average is computed over files, so one averaged value per feature/bin/region.

    Args:
        graph_data: Dict[str, Dict[str, List[float] | int]]. From prepare_chromscore_per_biospecimen_data.
        cell_types: List[str]|None. List of cell types to plot.

    Returns:
        pd.DataFrame. Dataframe with scipy.stats results objects.
    """
    data = copy.deepcopy(graph_data)
    if not cell_types:
        cell_types = list(data.keys())
        cell_types.remove("all_files")

    colors = px.colors.qualitative.Dark24[0:2]

    fig = go.Figure()
    if do_subplots:
        fig = make_subplots(
            rows=4,
            cols=4,
            shared_yaxes=True,
            vertical_spacing=0.075,
            horizontal_spacing=0.025,
            y_title="Average of max value in selected regions of 100kb (over files)",
        )

    # Filter
    try:
        data = {biospecimen: graph_data[biospecimen] for biospecimen in cell_types}
    except KeyError as err:
        raise KeyError(
            f"A cell type is missing from the graph_data.\ncell types: {graph_data.keys()}.\nDesired: {cell_types}."
        ) from err

    all_tests = []
    trace_names = []
    for idx, (biospecimen, data) in enumerate(data.items()):
        if biospecimen not in cell_types:
            continue

        avg_per_bin: List[float] = data["avg_per_bin"]  # type: ignore
        all_means_file_subset: List[float] = data["all_means_file_subset"]  # type: ignore

        nb_files = data["nb_files"]
        nb_features = len(avg_per_bin)

        if do_subplots:
            placement_dict = {
                "row": idx // 4 + 1,
                "col": idx % 4 + 1,
            }
        else:
            placement_dict = {}

        # Important features
        show_points = False
        if len(avg_per_bin) <= 10:
            show_points = "all"
        fig.add_trace(
            go.Violin(
                side="negative",
                name=f"trace{idx}",
                y=avg_per_bin,
                fillcolor=colors[0],
                line=dict(color="black", width=1.5 if do_subplots else 0),
                marker_size=3,
                jitter=0.1,
                pointpos=-0.4,
                showlegend=False,
                meanline_visible=True,
                points=show_points,
                spanmode="hard",
                legendgroup="All features",
                box=dict(
                    visible=True,
                    fillcolor=colors[0] if do_subplots else "black",
                    width=0.4,
                    line_width=0.5 if do_subplots else 0,
                ),
                scalemode="width",  # occupy all possible space for subplots
                scalegroup=f"trace{idx}",
            ),
            **placement_dict,  # type: ignore
        )

        # Global distribution comparison
        fig.add_trace(
            go.Violin(
                side="positive",
                name=f"trace{idx}",
                y=all_means_file_subset,
                fillcolor=colors[1],
                line=dict(color="black", width=1.5 if do_subplots else 0),
                showlegend=False,
                meanline_visible=True,
                points=False,
                spanmode="hard",
                legendgroup="All features",
                box=dict(
                    visible=True,
                    fillcolor=colors[1] if do_subplots else "black",
                    width=0.4,
                    line_width=0.5 if do_subplots else 0,
                ),
                scalemode="width",
                scalegroup=f"trace{idx}",
            ),
            **placement_dict,  # type: ignore
        )

        welchtest, bmtest, cohen_d = test_distribution(
            x=avg_per_bin,
            y=all_means_file_subset,
            verbose=False,
        )
        pvals = [welchtest.pvalue, bmtest.pvalue]
        if verbose:
            print(f"{biospecimen}, {nb_features} features, {nb_files} files")
            print(f"pvals [Welch, BM]: {pvals}\n\n")

        all_tests.append(
            [
                biospecimen,
                nb_files,
                nb_features,
                len(all_means_file_subset),
                welchtest,
                bmtest,
                cohen_d,
            ]
        )

        pval = float(np.max(pvals))
        pval_symbol = define_pval_label(pval * 16)

        if do_subplots:
            group_name = f"{biospecimen}<br>({nb_files} files, {nb_features} features)<br>p{pval_symbol}"
            fig.update_xaxes(
                showticklabels=False,
                row=idx // 4 + 1,
                col=idx % 4 + 1,
                title=group_name,
                title_standoff=2,
                title_font=dict(size=10),
            )
        else:
            group_name = f"{biospecimen} ({nb_files} files, {nb_features} features), p{pval_symbol}"
            trace_names.append(group_name)

    # Manually set names for traces
    if not do_subplots:
        newnames = {f"trace{idx}": name for idx, name in enumerate(trace_names)}
        fig.for_each_trace(lambda t: t.update(name=newnames[t.name]))

    # Legend with dummy points
    for i, name in enumerate(["Important SHAP features", "All features"]):
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=name,
                legendgroup=name,
                showlegend=True,
                marker=dict(color=colors[i], symbol="square"),
            ),
        )

    fig.update_yaxes(range=[0, 1])

    fig.update_layout(
        title="ChromScore per biospecimen file subset",
        width=1000,
        height=900,
        legend=dict(
            itemsizing="constant",
            yanchor="top",
            xanchor="right",
            y=1.1,
            x=0.8,
        ),
    )

    # fig.update_layout(violingap=0, violinmode='overlay')

    if not do_subplots:
        fig.update_layout(
            yaxis_title="Average of max value in selected regions of 100kb (over files)",
            xaxis_title="Biospecimen",
            width=700,
            height=700,
        )

    fig.show()

    if logdir is not None:
        print("Saving figure.")
        fig.write_image(logdir / f"{filename}.svg")
        fig.write_image(logdir / f"{filename}.png", scale=1.5)
        fig.write_html(logdir / f"{filename}.html")

    return pd.DataFrame(
        all_tests,
        columns=[
            "biospecimen",
            "nb_files",
            "Nb features (N_1)",
            "Nb features global (N_2)",
            "test_Welch",
            "test_BM",
            "cohen_d_effect_size",
        ],
    )

In [ ]:
def plot_chromscore_global_violin(
    graph_data: Dict[str, Dict[str, List[float] | int]],
    cell_types: List[str] | None = None,
    logdir: Path | None = None,
    filename: str = "chromscore_global_violin",
) -> List[Any]:
    """
    Plot boxplots for important features with their cell type files subset
    vs all features for all files.

    Args:
        graph_data: Dict[str, Dict[str, List[float] | int]]. From prepare_chromscore_per_biospecimen_data.
        cell_types: List[str]|None. List of cell types to plot.

    """
    data = copy.deepcopy(graph_data)
    if not cell_types:
        cell_types = list(data.keys())
        cell_types.remove("all_files")

    colors = px.colors.qualitative.Dark24[0:2]

    fig = go.Figure()

    # Filter
    try:
        data = {biospecimen: graph_data[biospecimen] for biospecimen in cell_types}
    except KeyError as err:
        raise KeyError(
            f"A cell type is missing from the graph_data.\ncell types: {graph_data.keys()}.\nDesired: {cell_types}."
        ) from err

    important_features_vals = []
    for _, data in enumerate(data.values()):
        avg_per_bin: List[float] = data["avg_per_bin"]  # type: ignore
        important_features_vals.extend(avg_per_bin)
    N_subsets = len(important_features_vals)

    # Important features
    fig.add_trace(
        go.Violin(
            name="trace",
            legendgroup="Important features (SHAP)",
            side="negative",
            y=important_features_vals,
            fillcolor=colors[0],
            line=dict(color="black", width=1.5),
            showlegend=False,
            meanline_visible=True,
            points=False,
            spanmode="hard",
            box=dict(
                visible=True,
                fillcolor=colors[0],
                width=0.4,
                line_width=1,
            ),
        ),
    )

    # Global distribution comparison
    big_N = len(graph_data["all_files"])
    fig.add_trace(
        go.Violin(
            name="trace",
            legendgroup="All features",
            side="positive",
            y=graph_data["all_files"],
            fillcolor=colors[1],
            line=dict(color="black", width=1.5),
            showlegend=False,
            meanline_visible=True,
            points=False,
            spanmode="hard",
            box=dict(
                visible=True,
                fillcolor=colors[1],
                width=0.4,
                line_width=1,
            ),
        ),
    )

    welchtest, bmtest, effect_size = test_distribution(
        x=important_features_vals,
        y=graph_data["all_files"],  # type: ignore
        verbose=True,
    )

    stats = [
        "Combined classes",
        total_files,
        N_subsets,
        big_N,
        welchtest,
        bmtest,
        effect_size,
    ]

    # Legend with dummy points
    for i, name in enumerate(
        [f"Important SHAP features ({N_subsets})", f"All features ({big_N})"]
    ):
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=name,
                legendgroup=name.split("(")[0].strip(),
                showlegend=True,
                marker=dict(color=colors[i], symbol="square"),
            ),
        )

    fig.update_xaxes(showticklabels=False)

    fig.update_yaxes(range=[0, 1])

    fig.update_layout(
        title_text="Average of max chromscore<br>in selected regions of 100kb (over files)",
        title_xanchor="center",
        title_x=0.5,
        width=500,
        height=500,
        legend=dict(yanchor="top", y=-0.01, xanchor="center", x=0.5),
    )

    fig.show()

    if logdir is not None:
        print("Saving figure.")
        fig.write_image(logdir / f"{filename}.svg")
        fig.write_image(logdir / f"{filename}.png", scale=1.5)
        fig.write_html(logdir / f"{filename}.html")

    return stats

In [ ]:
logdir = paper_dir / "figures" / "chromscore"
if not logdir.exists():
    logdir.mkdir(parents=True)

In [ ]:
# Sanity check, missing values
groupby = merged_df.groupby(CELL_TYPE)["chromscore_value"].apply(lambda x: x.isna().sum())
if not groupby.sum() == 0:
    display(groupby)
    raise ValueError("Missing values in chromscore_value")

This next cell can take more than 1 minute.

In [ ]:
graph_data = prepare_chromscore_per_biospecimen_data(
    selected_bins_df=merged_df, all_chromscores_df=melted_chromscores
)

In [ ]:
logdir = paper_dir / "figures" / "chromscore" / "max"
stats_df = plot_chromscore_per_biospecimen_violin(
    graph_data=graph_data,
    do_subplots=True,
    logdir=logdir,
    filename="v6_important_features_16ct_max_chromscore_100kb_per_biospecimen_2violin_with_points",
)

In [ ]:
stats_global = plot_chromscore_global_violin(
    graph_data=graph_data,
)

In [ ]:
final_stats_df = pd.concat(
    [
        stats_df,
        pd.DataFrame([stats_global], columns=stats_df.columns),
    ],
    ignore_index=True,
)

### Graph statistics

In [ ]:
def detail_statistics(df: pd.DataFrame) -> pd.DataFrame:
    """Create dataframe with detailed statistics from test objects."""
    new_df = df.copy().set_index("biospecimen")

    # Common statistics + p-value corrections
    for test in ["Welch", "BM"]:
        new_df[f"{test}_pval"] = new_df[f"test_{test}"].apply(lambda x: x.pvalue)
        new_df[f"{test}_pval_corr"] = new_df[f"{test}_pval"].apply(
            lambda x: min(x * 16, 1.0)
        )
        new_df[f"{test}_stat"] = new_df[f"test_{test}"].apply(lambda x: x.statistic)

    # Additional Welch statistics
    # Degrees of freedom and 95% confidence interval
    new_df["Welch_DoF"] = new_df["test_Welch"].apply(lambda x: x.df)

    new_df["Welch_conf_int_95"] = new_df["test_Welch"].apply(
        lambda x: [f"{v:.4f}" for v in x.confidence_interval()]
    )

    # Final max p-value corrected
    cols = [label for label in new_df.columns if "pval_corr" in label]
    new_df["max_pval_corr"] = new_df[cols].max(axis=1)

    # Drop original test columns
    for test in ["Welch", "BM"]:
        new_df = new_df.drop(columns=[f"test_{test}"])

    # Formatting
    new_df.sort_values(by="nb_files", ascending=True, inplace=True)
    new_df.rename(
        columns={
            "nb_files": "Nb EpiRR/Samples",
            "cohen_d_effect_size": "Cohen_d",
        },
        inplace=True,
    )

    return new_df

In [ ]:
out_df = detail_statistics(final_stats_df)
display(out_df)

In [ ]:
fig_dir = paper_dir / "figures" / "chromscore" / "max"
out_df.to_csv(
    fig_dir / "max_chromscore_important_bins_per_biospecimen_stats.tsv",
    sep="\t",
    index=True,
    float_format="%.4e",
)